# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a complete walkthrough for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
  
`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"\nDataset Identifier: {metadata.identifier}")
print(f"Version: {metadata.version}")
print(f"Published: {metadata.date_published}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

With `mlcroissant`, you can list the record sets and inspect their fields before loading data. Each entity (record set, field, column) is referenced by its `@id` field.

In [ ]:
# List all record sets and their fields, referencing them by '@id'
record_sets = list(dataset.record_sets)
if not record_sets:
    print("No record sets found in dataset. Try using .record_sets to explore.")
else:
    for rs in record_sets:
        print(f"\nRecord Set: {rs.name}")
        print(f"  @id: {rs.id}")
        print("  Fields:")
        for field in rs.fields:
            print(f"    - {field.name} (@id: {field.id}) [type: {field.data_type}]")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

*If there are multiple record sets, all will be loaded.*

In [ ]:
# Extract data from each record set
# Reference record sets and fields by their @id
dataframes = {}
for rs in dataset.record_sets:
    records = list(dataset.records(record_set=rs.id))
    df = pd.DataFrame(records)
    dataframes[rs.id] = df
    print(f"Loaded record set: {rs.name} (@id: {rs.id}) with {len(df)} rows and columns:")
    print(df.columns.tolist())
    print('---')

# For demonstration, pick the first record set for EDA
if dataset.record_sets:
    primary_record_set_id = dataset.record_sets[0].id
    print(f"\nDisplaying first 5 records from the primary record set (@id: {primary_record_set_id}):")
    display(dataframes[primary_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming data distributions, and grouping data by key attributes to prepare it for further analysis.

> **Remember:** Always reference fields and record sets by their `@id`.

In [ ]:
# EDA on the chosen record set (@id: primary_record_set_id)
df = dataframes[primary_record_set_id]

# Find a numeric field to analyze (select the first numeric-looking column by dtype)
import numpy as np
numeric_fields = df.select_dtypes(include=[np.number]).columns.tolist()

if numeric_fields:
    numeric_field_id = numeric_fields[0]
    print(f"Using numeric field: {numeric_field_id} for analysis")
    threshold = df[numeric_field_id].mean()  # As demo, use mean as threshold

    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (mean):")
    display(filtered_df.head())

    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, norm_col]].head())

    # Select a group-by field with relatively few unique values (e.g., 2-20)
    group_field = None
    for col in df.columns:
        if col != numeric_field_id and df[col].nunique() > 1 and df[col].nunique() < 20:
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().to_frame(name=f"mean_{numeric_field_id}")
        print(f"\nGrouped mean {numeric_field_id} by {group_field}:")
        display(grouped_df.head())
    else:
        print("No suitable grouping field found for this record set.")
else:
    print("No numeric field found for EDA.\nAvailable columns:", df.columns.tolist())

## 5. Visualization
Visualize the distribution of numeric fields or relationships between fields in the dataset.

In [ ]:
# Basic visualization of the numeric field and (if available) group-by field
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_fields:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    if group_field:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=df[group_field], y=df[numeric_field_id])
        plt.title(f'{numeric_field_id} by {group_field}')
        plt.xlabel(group_field)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("No numeric field available to plot.")

## 6. Conclusion
This exploration demonstrates how to use `mlcroissant` to programmatically load, inspect, and analyze a standardized FAIR² Croissant dataset. Key steps include:
- Accessing dataset metadata and record sets using Croissant schema and referencing every entity by its `@id`,
- Loading tabular data into familiar data analysis tools like pandas,
- Applying EDA techniques to numeric and categorical fields,
- Visualizing data distributions and groupings.

For more advanced analysis or handling of additional record sets and fields, you can further inspect the schema structure via the dataset's `.record_sets` and `.fields` attributes—using the `@id` for uniquely identifying and referencing each entity.
